In [73]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
import pandas as pd

In [102]:
df=pd.read_csv('C:/Users/Lenovo/Downloads/data.csv')

In [75]:
df.head()

,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,sqft_above,sqft_basement,yr_built,yr_renovated,street,city,statezip,country
0,2014-05-02 00:00:00,313000.0,3.0,1.50,1340,7912,1.5,0,0,3,1340,0,1955,2005,18810 Densmore Ave N,Shoreline,WA 98133,USA
1,2014-05-02 00:00:00,2384000.0,5.0,2.50,3650,9050,2.0,0,4,5,3370,280,1921,0,709 W Blaine St,Seattle,WA 98119,USA
2,2014-05-02 00:00:00,342000.0,3.0,2.00,1930,11947,1.0,0,0,4,1930,0,1966,0,26206-26214 143rd Ave SE,Kent,WA 98042,USA
3,2014-05-02 00:00:00,420000.0,3.0,2.25,2000,8030,1.0,0,0,4,1000,1000,1963,0,857 170th Pl NE,Bellevue,WA 98008,USA
4,2014-05-02 00:00:00,550000.0,4.0,2.50,1940,10500,1.0,0,0,4,1140,800,1976,1992,9105 170th Ave NE,Redmond,WA 98052,USA


In [76]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4600 entries, 0 to 4599
Data columns (total 18 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   date           4600 non-null   str    
 1   price          4600 non-null   float64
 2   bedrooms       4600 non-null   float64
 3   bathrooms      4600 non-null   float64
 4   sqft_living    4600 non-null   int64  
 5   sqft_lot       4600 non-null   int64  
 6   floors         4600 non-null   float64
 7   waterfront     4600 non-null   int64  
 8   view           4600 non-null   int64  
 9   condition      4600 non-null   int64  
 10  sqft_above     4600 non-null   int64  
 11  sqft_basement  4600 non-null   int64  
 12  yr_built       4600 non-null   int64  
 13  yr_renovated   4600 non-null   int64  
 14  street         4600 non-null   str    
 15  city           4600 non-null   str    
 16  statezip       4600 non-null   str    
 17  country        4600 non-null   str    
dtypes: float64(4), int6

In [103]:
#feature engineering
df['date'] = pd.to_datetime(df['date'])
print(df['date'].dt.year.value_counts())
df=df.drop(['street','date'], axis =1)



date
2014    4600
Name: count, dtype: int64


In [104]:
#feature / trget split
X=df.drop('price', axis=1)
y=df['price']

In [106]:
#continue feature engineering
encoder = OneHotEncoder(handle_unknown='ignore')
X_scalar = StandardScaler()
y_scalar= StandardScaler()

preprocessor = ColumnTransformer([
    ('num', X_scalar, ['bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors',
       'waterfront', 'view', 'condition', 'sqft_above', 'sqft_basement',
       'yr_built', 'yr_renovated']),
    
    ('cat', encoder, [ 'city', 'statezip', 'country'])
    
])


In [ ]:
from sklearn.model_selection import train_test_split

X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(X, y, test_size=0.2, random_state=5)

X_train = preprocessor.fit_transform(X_train_raw)   
X_test = preprocessor.transform(X_test_raw)          

y_train = y_scalar.fit_transform(y_train_raw.values.reshape(-1,1))
y_test = y_scalar.transform(y_test_raw.values.reshape(-1,1))

For feature engineering :
1- i seperated the year from the date feaure and then notice that they are all the same year so that won't be an important feauture in training that's why i dropped that column.

2- i did a scalar(to scale the numerical feautures in the data frame so they become in the same range best for training) and an encoder (to encode the categorical features in the dataframe )



In [95]:
df.columns

Index(['price', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors',
       'waterfront', 'view', 'condition', 'sqft_above', 'sqft_basement',
       'yr_built', 'yr_renovated', 'street', 'city', 'statezip', 'country'],
      dtype='str')

In [108]:
# model from Week 3
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np


model =DecisionTreeRegressor(max_depth=5,random_state=42)
model.fit(X_train,y_train)
predictions =model.predict(X_test)

baseline_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='r2')

print("R2",baseline_scores)
print('R2 Mean', baseline_scores.mean())


R2 [0.02234558 0.01528135 0.46980663 0.40687869 0.55686535]
R2 Mean 0.2942355193308968


In [109]:
# hyperparameter grid for Desicion Tree
param_grid = {
    'criterion': ['squared_error'],     # split quality measure
    'max_depth': [3, 5, 7, 10],        # max depth of the tree
    'min_samples_split': [20, 40, 60, 100],       # min samples to split a node
    'min_samples_leaf': [10, 20, 30, 50],         # min samples allowed in a leaf
    'max_features': [None, 'sqrt', 'log2']     # features considered per split
}

In [110]:
# GridSearchCV
from sklearn.model_selection import GridSearchCV
grid =GridSearchCV(DecisionTreeRegressor(random_state=42),param_grid, cv =5, n_jobs=-1, verbose=2, scoring='r2')
grid.fit(X_train, y_train)
print("Best Parameters",grid.best_params_)
print("Best score",grid.best_score_)
best_model =grid.best_estimator_

Fitting 5 folds for each of 192 candidates, totalling 960 fits
Best Parameters {'criterion': 'squared_error', 'max_depth': 10, 'max_features': None, 'min_samples_leaf': 10, 'min_samples_split': 60}
Best score 0.3119802131418682


In [111]:
# comparing scores
print("CV R² (best_score_):", grid.best_score_)
print("Train R²:", best_model.score(X_train, y_train))
print("Test R²:", best_model.score(X_test, y_test))

CV R² (best_score_): 0.3119802131418682
Train R²: 0.25508071209232286
Test R²: 0.45455962500585845


`criterion` was tested with both `squared_error` and `absolute_error`. Since 
`squared_error` won in every trial with identical best scores, and `absolute_error` is significantly more expensive to compute, it was excluded from the final grid 
to reduce runtime without sacrificing performance.

In [112]:
#most important features

feature_names = preprocessor.get_feature_names_out()  # replace 'preprocessor' with your actual transformer/pipeline variable name

importances = pd.Series(best_model.feature_importances_, index=feature_names)
importances_sorted = importances.sort_values(ascending=False).head()
print(importances_sorted)

num__sqft_living          0.695416
num__yr_built             0.049544
cat__statezip_WA 98199    0.046447
cat__statezip_WA 98031    0.037650
cat__city_Seattle         0.031521
dtype: float64


In [113]:
results_df = pd.DataFrame(grid.cv_results_)

# Example: average CV score for each value of a given hyperparameter, across all other combos
for param in ['param_max_depth', 'param_min_samples_split', 'param_min_samples_leaf', 'param_max_features']:
    print(f"\n--- Effect of {param} ---")
    print(results_df.groupby(param)['mean_test_score'].mean().sort_values(ascending=False))


--- Effect of param_max_depth ---
param_max_depth
10    0.201259
7     0.187962
5     0.154040
3     0.134774
Name: mean_test_score, dtype: float64

--- Effect of param_min_samples_split ---
param_min_samples_split
40     0.172293
100    0.169928
60     0.168905
20     0.166909
Name: mean_test_score, dtype: float64

--- Effect of param_min_samples_leaf ---
param_min_samples_leaf
30    0.178328
10    0.173622
20    0.167712
50    0.158372
Name: mean_test_score, dtype: float64

--- Effect of param_max_features ---
param_max_features
sqrt    0.139287
log2    0.103777
Name: mean_test_score, dtype: float64


### Step 5: Most Impactful Feature & Hyperparameter

**Most impactful engineered feature**: 'num__sqft_living', with an importance score of 
0.69 — the highest among engineered features

**Most impactful hyperparameter**: `max_depth`, whose values produced the largest 
swing in mean CV R² ,tree depth was the dominant lever controlling model performance on this dataset.